<a href="https://colab.research.google.com/github/ghalde194/APP1/blob/main/Unificacao_Excel_o_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from google.colab import files
import pandas as pd
from datetime import datetime
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
print("🚀 PROGRAMA - Unificacao Premium e Homologacao de Dados")
print("=" * 90)
# 1. Upload do arquivo
uploaded = files.upload()
nome_arquivo = next(iter(uploaded))
# 2. Ler as abas
df_datasul = pd.read_excel(nome_arquivo, sheet_name="Datasul ")
df_workday = pd.read_excel(nome_arquivo, sheet_name="Workday")
print(f"✅ Datasul carregado: {len(df_datasul)} linhas")
print(f"✅ Workday carregado: {len(df_workday)} linhas")
# 3. Limpeza e Padronizacao da Matricula
# O .replace(".0", "") evita que matrículas lidas como float virem '1234.0'
df_datasul["Matrícula"] = df_datasul["Matrícula"].astype(str).str.replace(".0", "", regex=False).str.strip()
df_workday["ID DATASUL"] = df_workday["ID DATASUL"].astype(str).str.replace(".0", "", regex=False).str.strip()
# 4. Cruzamento TOTAL (Full Outer Merge)
df_final = df_datasul.merge(
    df_workday,
    left_on="Matrícula",
    right_on="ID DATASUL",
    how="outer",
    suffixes=("", "_Workday")
)
# 5. Inteligencia de Preenchimento Cruzado de Funcionarios
df_final["Matrícula"] = df_final["Matrícula"].fillna(df_final["ID DATASUL"])
if "Legal Name in General Display Format" in df_final.columns:
    df_final["Nome"] = df_final["Nome"].fillna(df_final["Legal Name in General Display Format"])
# 6. Mapeamento para Nome Completo do Gestor
mapa_nomes = pd.Series(df_final["Nome"].values, index=df_final["Matrícula"]).to_dict()
if "Manager ID" in df_final.columns:
    df_final["Manager ID"] = df_final["Manager ID"].astype(str).str.replace(".0", "", regex=False).str.strip()

    # Criamos a nova coluna de Gestor trazendo o nome completo baseado no Manager ID
    df_final["Gestor (Nome Completo)"] = df_final["Manager ID"].map(mapa_nomes)

    # Se o gestor nao for mapeado na base local, mantemos o que estava no Workday
    df_final["Gestor (Nome Completo)"] = df_final["Gestor (Nome Completo)"].fillna(df_final["Worker's Manager(s)"])
# 7. Organizacao e Selecao de Colunas Finais
colunas_organizadas = [
    "Matrícula",
    "Nome",
    "Sexo",
    "Cargo Básico-Descrição",
    "Local Pgto",
    "Local Pgto-Descrição",
    "Data Admissão",
    "ID WORKDAY",
    "Continuous Service Date",
    "Employee Type",
    "Active Status with Date",
    "Job Code",
    "Job Profile (Primary)",
    "Business Title",
    "Compensation Grade",
    "Location",
    "Company",
    "Cost Centers",
    "Manager ID",
    "Gestor (Nome Completo)",
    "Compensation Most Recent Change Date",
    "Compensation Plan",
    "Job Family Group",
    "Job Family",
    "Management Level",
    "Email Primary Work or Primary Home"
]
colunas_finais = [col for col in colunas_organizadas if col in df_final.columns]
df_final = df_final[colunas_finais].copy()
# Remove quaisquer espacos extras nas bordas dos textos antes de preencher os nulos
for col in df_final.select_dtypes(include=['object']).columns:
    df_final[col] = df_final[col].astype(str).str.strip()
# Substitui lacunas finais por "-"
df_final = df_final.fillna("-")
# 7.5. Ordena o DataFrame final por nome em ordem alfabética
df_final = df_final.sort_values(by='Nome', ascending=True).reset_index(drop=True)
print(f"\n✅ Planilha final gerada com {len(df_final)} linhas unicas.")
# 8. Salvar e Aplicar Formatacao Avancada no Excel
nome_saida = f"Planilha_Mapeamento_Final_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
# Adicionado datetime_format para garantir que as datas fiquem legíveis no Excel
with pd.ExcelWriter(nome_saida, engine="openpyxl", datetime_format="DD/MM/YYYY") as writer:
    df_final.to_excel(writer, sheet_name="Base Unificada", index=False)
    ws = writer.sheets["Base Unificada"]
    # Ativa as linhas de grade do Excel
    ws.views.sheetView[0].showGridLines = True
    # --- ESTILOS ---
    cor_cabecalho = PatternFill(start_color="1A365D", end_color="1A365D", fill_type="solid")
    cor_zebra = PatternFill(start_color="F8FAFC", end_color="F8FAFC", fill_type="solid")
    fonte_cabecalho = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
    fonte_corpo = Font(name="Segoe UI", size=10)
    alinhar_centro = Alignment(horizontal="center", vertical="center")
    alinhar_esquerda = Alignment(horizontal="left", vertical="center")
    borda_fina = Side(border_style="thin", color="E2E8F0")
    borda_celula = Border(left=borda_fina, right=borda_fina, top=borda_fina, bottom=borda_fina)
    # Formatacao do Cabecalho
    for cell in ws[1]:
        cell.fill = cor_cabecalho
        cell.font = fonte_cabecalho
        cell.alignment = alinhar_centro
        cell.border = borda_celula
    # Formatacao do Corpo da Planilha
    for r_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=2):
        for cell in row:
            cell.font = fonte_corpo
            cell.border = borda_celula
            cell.alignment = alinhar_esquerda
            # Centraliza o traco de preenchimento
            if cell.value == "-":
                cell.alignment = alinhar_centro
            # Centraliza colunas de identificacao e datas
            nome_coluna = ws.cell(row=1, column=cell.column).value

            # Força colunas de matrícula a serem tratadas puramente como TEXTO pelo Excel
            if "Matrícula" in str(nome_coluna) or "ID" in str(nome_coluna):
                cell.number_format = "@"

            if any(term in str(nome_coluna) for term in ["Matrícula", "Date", "Data", "ID", "Sexo", "Status"]):
                cell.alignment = alinhar_centro
            # Efeito Zebra
            if r_idx % 2 == 0:
                cell.fill = cor_zebra
    # Ajuste automatico de largura considerando o tamanho do CABECALHO e do CONTEUDO
    for col in ws.columns:
        column_letter = col[0].column_letter
        # Mede o maior comprimento entre o título da coluna e os dados dela
        max_len_conteudo = max(len(str(cell.value or '')) for cell in col)

        ws.column_dimensions[column_letter].width = min(max_len_conteudo + 4, 45)
    # Congelar o cabecalho
    ws.freeze_panes = "A2"
print(f"\n✅ Arquivo gerado com sucesso: {nome_saida}")
files.download(nome_saida)


🚀 PROGRAMA - Unificacao Premium e Homologacao de Dados


Saving Relatórios Funcionários 300326 - Leonardo.xlsx to Relatórios Funcionários 300326 - Leonardo (12).xlsx
✅ Datasul carregado: 869 linhas
✅ Workday carregado: 839 linhas

✅ Planilha final gerada com 921 linhas unicas.

✅ Arquivo gerado com sucesso: Planilha_Mapeamento_Final_20260402_1234.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>